### PPO vs. CISPO — A Reality Check 😂

- **PPO:** Once the ratio exceeds the clipping boundary, the corresponding surrogate gradient becomes **zero**.
- **CISPO:** Clip the importance weight instead, **preserving a nonzero gradient**.
- **Irony:** I independently proposed the CISPO intuition this morning, while AI and I spent ages discussing why PPO's zero-gradient clipping was beneficial.
- **Takeaway:** Many RL optimization tricks are empirically motivated design choices, not mathematically proven optimal solutions. Papers often formalize these heuristics with equations.

**Lesson learned:** Understand the math, but don't worship the algorithms. RL engineering still involves a lot of alchemy. 😂

---
1 question to ask whether a new PO paper is worth reading:

>Does it help me understand a failure mode I didn’t know about before?
>
>or does it just tell me that tweaking some hyperparameter boosted the AIME score by three points?

---

# GSPO — Group Sequence Policy Optimization

## 1. The Problem: Why Clip Individual Tokens When Reward Is Assigned to the Whole Sequence?

Suppose a model generates a 10-token completion and receives a positive advantage $A=1$.

The importance ratios are:

$$
\rho=[1.5,1,1,1,1,1,1,1,1,1]
$$

With a clipping range of $[0.8,1.2]$:

- **GRPO (token-level clipping):** The first token is clipped, so its surrogate gradient becomes zero. The remaining nine tokens still contribute gradients.
- **GSPO (sequence-level clipping):** Instead of clipping individual tokens, calculate one ratio for the entire sequence.

The question becomes:

**Can we use the overall change in sequence probability rather than letting individual token ratios determine clipping?**

## 2. The Core Idea: Sequence-Level Importance Ratio

For each token:

$$
\rho_t=\frac{\pi_\theta(y_t\mid x,y_{<t})}{\pi_{\mathrm{old}}(y_t\mid x,y_{<t})}
$$

The mathematically exact sequence importance ratio is:

$$
\rho_{\mathrm{seq}}=\prod_{t=1}^{T}\rho_t
$$

But multiplying many ratios can produce extreme values.

For example, suppose every token has a ratio of $1.01$:

$$
\rho_{\mathrm{seq}}=1.01^{500}\approx144.77
$$

Each token's probability increased by only 1%, yet the full sequence ratio is approximately 145!

**GSPO's solution: Use the geometric mean of token ratios.**

$$
\boxed{
s=\left(\prod_{t=1}^{T}\rho_t\right)^{1/T}
=\exp\left(\frac1T\sum_t\log\rho_t\right)
}
$$

For the 500-token example:

$$
s=(1.01^{500})^{1/500}=1.01
$$

This makes the ratio's numerical scale much less sensitive to sequence length.

Returning to our original 10-token example:

$$
s=(1.5\times1^9)^{1/10}\approx1.0414
$$

With the illustrative clipping range $[0.8,1.2]$, the sequence is not clipped.

**Key difference:**

- GRPO: Each token has its own ratio and clipping decision.
- GSPO: All tokens in a sequence share one length-normalized sequence ratio and clipping decision.

GSPO uses this ratio in a sequence-level PPO-style surrogate objective.

## 3. The Trade-off: Mathematical Exactness vs. Training Stability

The geometric mean is **not the exact importance sampling ratio**.

Consider two possible completions, each containing two tokens:

| Completion | Reward | Old Probability | New Probability |
|---|---:|---:|---:|
| `a4` | 1 | 0.25 | 0.50 |
| `b6` | 0 | 0.75 | 0.50 |

The true expected reward under the new policy is:

$$
J_{\mathrm{new}}=0.50
$$

Using exact sequence importance sampling:

$$
J_{\mathrm{IS}}=0.25\times2\times1=0.50
$$

Using the GSPO geometric ratio instead:

$$
J_{\mathrm{geometric}}=0.25\times\sqrt2\times1\approx0.354
$$

**The geometric ratio no longer preserves the unbiased importance sampling identity.**

This does not mean GSPO's actual training reward will equal 0.354. It demonstrates that its length-normalized weight cannot directly replace the exact likelihood ratio in an unbiased reward estimator.

Another limitation: if two token ratios are $[2,0.5]$, their geometric mean is $1$, even though both conditional token probabilities changed substantially.

GSPO therefore makes a deliberate trade-off:

- **Benefit:** More stable ratio scaling for long sequences, less sensitivity to isolated token-level fluctuations, and shared sequence-level clipping.
- **Cost:** No general unbiased importance sampling guarantee; some token-level changes can be hidden by aggregation.

**Core insight:** GSPO prioritizes a more practical sequence-level optimization surrogate over mathematically exact sequence importance weighting.

---

# LLM RL Algorithms as LEGO Pieces

$$
\boxed{\text{Policy Optimizer}=(A,I,N,C,S,L)}
$$

- **A — Advantage Estimation:** How do we determine whether a sampled completion is better than expected?
- **I — Importance Ratio:** How do we measure changes between old and new policies?
- **N — Normalization:** How do we scale advantages or other optimization quantities?
- **C — Clipping:** How do we constrain the influence of large policy changes?
- **S — Sampling:** Which prompts and completions enter training?
- **L — Loss Aggregation:** How are token and sequence losses combined?

| Algorithm | Main LEGO Changes | Core Idea |
|---|---|---|
| **REINFORCE** | A | Weight the log-probability gradient by reward. |
| **RLOO** | A, S | Sample multiple completions for the same prompt; use the other completions as a leave-one-out baseline. |
| **PPO** | I, C | Importance ratios with a clipped surrogate objective. |
| **GRPO** | A, N, C | Group-centered, standard-deviation-normalized advantages with PPO-style updates. |
| **Dr. GRPO** | N, L | Remove group standard deviation normalization and use fixed-length loss normalization. |
| **DAPO** | S, C, L | Dynamic sampling, Clip-Higher, token-level loss aggregation, and overlong-response handling. |
| **CISPO** | I, C | Clip importance weights while preserving the log-probability gradient. |
| **GSPO** | I, C, L | Sequence-level ratio, sequence-level clipping, and sequence-level surrogate objective. |

**Final takeaway:** These algorithms largely recombine a small set of design choices. But the LEGO pieces are not mathematically independent: changing one piece can alter the gradients, optimization stability, and theoretical guarantees of the entire algorithm.